# Lab 13: Free APIs and Integration - The Complete Digital Twin

## Putting it together

This lab turns your own bio into a chatbot persona that remembers the chat. We call it a "digital twin".

The cells below set up `call_groq`. Then you write `build_digital_twin_system_prompt(profile)`. At the end you run a real chat with many turns. The chat keeps adding to a `messages` list, one turn at a time.

### Setup - get a free key at https://console.groq.com/keys

In [ ]:
"""Get your free Groq API key at https://console.groq.com/keys - it's never written to this notebook."""
import getpass
import json
import requests

GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
DEFAULT_MODEL = "llama-3.1-8b-instant"

GROQ_API_KEY = getpass.getpass("Enter your GROQ_API_KEY: ")


def build_request_payload(messages, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The request BODY: what we're asking the model to do."""
    payload = {"model": model, "messages": messages, "temperature": temperature}
    if response_format is not None:
        payload["response_format"] = response_format
    return payload


def extract_reply(response_json):
    """Pulls the assistant's text out of the RESPONSE body."""
    return response_json["choices"][0]["message"]["content"]


def call_groq(messages, api_key=None, model=DEFAULT_MODEL, temperature=0.7, response_format=None):
    """The full request: ENDPOINT + METHOD (POST) + HEADERS (auth) + BODY -> RESPONSE."""
    headers = {
        "Authorization": f"Bearer {api_key or GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = build_request_payload(messages, model=model, temperature=temperature, response_format=response_format)
    response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return extract_reply(response.json())


print("Groq setup ready. Call call_groq(messages) to send a chat completion request.")


## Exercise

### Goal
You write a function that turns a small dict about a person into a **system prompt**. A *system prompt* is the first message in a chat. It tells the model who to be and what rules to follow. For example: "answer as this person, in this tone, knowing these facts, staying inside these limits". You use this prompt in the project below to build your own "digital twin".

### `build_digital_twin_system_prompt(profile)`
Code shape (replace each `...`):
```python
def build_digital_twin_system_prompt(profile):
    lines = [
        f"... {profile['name']} ...",   # intro line
        ...,                            # tone line
        ...,                            # facts heading
    ]
    for fact in ...:
        lines.append(...)
    # boundaries heading, then one bullet per boundary
    return ...
```
**You get:** `profile`, a dict (like `{"name": "Ada"}`; you read a value with `profile["name"]`). It has exactly these keys:

- `name` (str), for example `"Ada"`
- `role` (str), for example `"a data analyst"`
- `tone` (str), for example `"warm, curious"`
- `facts` (list of str), true things about the person
- `boundaries` (list of str), things the twin must refuse or steer away from

**You give back:** one `str` with several lines, in this order:

1. A line saying the model is role-playing as that person. Use the name and role.
2. A line that says the speaking tone.
3. A heading line for facts. Then **one bullet line per fact**. A *bullet line* starts with `- `, like `- likes pandas`.
4. A heading line for boundaries. Then **one bullet line per boundary**.

You choose the exact words of the sentences. The bullets must start with `- `. Your function must work for any number of facts and boundaries (two, five, or even zero). Do not hard-code the count.

**Idea:** make a Python list of lines. Add the fixed lines first. Then add one bullet per fact and one per boundary (use a loop, or a list comprehension: a one-line loop like `[x * 2 for x in [1, 2]]`, which gives `[2, 4]`). A bullet can be built with an f-string, a string with values dropped in: if `name = "Ada"`, then `f"Hi {name}"` gives `"Hi Ada"`. At the end, join everything with `"\n"` (a newline character) and give that back. For example, `"\n".join(["a", "b"])` gives `"a\nb"`, which prints as two lines.

**Example.** For this profile:

```python
{"name": "Ada", "role": "a data analyst", "tone": "warm, curious",
 "facts": ["likes pandas", "runs weekly"], "boundaries": ["no medical advice"]}
```

one valid result, when printed, looks like this (it comes from running the reference solution):

```
You are role-playing as Ada, a data analyst.
Speak in a tone that is warm, curious.
Facts you can draw on about this person:
- likes pandas
- runs weekly
Boundaries you must respect:
- no medical advice
```

That is 7 lines: 2 intro lines, 1 facts heading, 2 fact bullets, 1 boundaries heading, 1 boundary bullet. With empty `facts` and `boundaries` lists you would get just 4 lines (the 2 intro lines and the 2 headings).

**Check:** `prompt.count("\n- ")` (the number of bullet lines) should equal `len(facts) + len(boundaries)`, so 3 for the Ada example. Also check that the result is a `str`, not a list, and that `"Ada"` appears in it.

### What the checks below test
The code cell builds a profile for "Vera" and calls your function. Then three `assert` lines check that the name `Vera`, the fact text `FastAPI`, and the boundary text `financial or legal advice` all appear somewhere in your prompt. An `assert` stops with an `AssertionError` and a message if its condition is false. When every assert passes, the cell prints **All checks passed.**

That only proves the key pieces are there. So also read your printed prompt yourself. Make sure the order and the bullets look right.

In [ ]:
def build_digital_twin_system_prompt(profile: dict) -> str:
    """Build a system prompt that tells the model to answer AS the person in `profile`.

    You get: profile, a dict with the keys "name" (str), "role" (str),
    "tone" (str), "facts" (list of str) and "boundaries" (list of str).

    You give back: one str with several lines: an intro line with the name
    and role, a tone line, a facts heading plus one "- " bullet per fact,
    and a boundaries heading plus one "- " bullet per boundary.

    Example:
        profile = {"name": "Ada", "role": "a data analyst", "tone": "warm, curious",
                   "facts": ["likes pandas"], "boundaries": ["no medical advice"]}
        build_digital_twin_system_prompt(profile) gives these 6 lines:
            You are role-playing as Ada, a data analyst.
            Speak in a tone that is warm, curious.
            Facts you can draw on about this person:
            - likes pandas
            Boundaries you must respect:
            - no medical advice

    TODO:
        1. Start a list with the intro line (name and role), the tone line,
           and a heading line for facts.
        2. Add one "- " bullet per fact (use a loop or a list comprehension).
        3. Add a heading line for boundaries, then one bullet per boundary.
        4. Join the list with a newline character and give back the resulting str.
    """
    raise NotImplementedError("build_digital_twin_system_prompt() is not implemented yet - replace the TODO with your code")


test_profile = {
    "name": "Vera",
    "role": "a backend engineer learning LLM foundations",
    "tone": "direct and a little dry",
    "facts": ["works primarily in Python and FastAPI", "is building a course on LLM foundations"],
    "boundaries": ["do not give financial or legal advice", "do not make up specific employers or dates"],
}
prompt = build_digital_twin_system_prompt(test_profile)
assert "Vera" in prompt, 'expected ' + repr("Vera") + ' to be in prompt, but it is ' + repr(prompt)
assert "FastAPI" in prompt, 'expected ' + repr("FastAPI") + ' to be in prompt, but it is ' + repr(prompt)
assert "financial or legal advice" in prompt, 'expected ' + repr("financial or legal advice") + ' to be in prompt, but it is ' + repr(prompt)

print("All checks passed.")

## Solution

In [ ]:
def build_digital_twin_system_prompt(profile: dict) -> str:
    lines = [
        f"You are role-playing as {profile['name']}, {profile['role']}.",
        f"Speak in a tone that is {profile['tone']}.",
        "Facts you can draw on about this person:",
    ]
    lines += [f"- {fact}" for fact in profile["facts"]]
    lines.append("Boundaries you must respect:")
    lines += [f"- {b}" for b in profile["boundaries"]]
    return "\n".join(lines)


print(build_digital_twin_system_prompt(test_profile))

## Real-world project (LIVE - needs your GROQ_API_KEY above)

Fill in `my_profile` below with your own real bio. Then run a chat with many turns against it. Watch `messages` grow after each turn. That is the whole trick.

In [ ]:
my_profile = {
    "name": "your name",
    "role": "however you'd describe your role in one clause",
    "tone": "a few adjectives for how you talk",
    "facts": [
        "a real fact about you",
        "another real fact about you",
    ],
    "boundaries": [
        "something this digital twin should always refuse to do",
    ],
}

In [ ]:
messages = [{"role": "system", "content": build_digital_twin_system_prompt(my_profile)}]

conversation_turns = [
    "Hi! What are you working on lately?",
    "What's your take on learning by building real projects vs. reading theory?",
]

for user_message in conversation_turns:
    messages.append({"role": "user", "content": user_message})
    reply = call_groq(messages)
    messages.append({"role": "assistant", "content": reply})
    print(f"You: {user_message}")
    print(f"Twin: {reply}\n")

print(f"Final messages list has {len(messages)} turns.")